# Evo Aggregated Statistics to MongoDB

This notebook calculates aggregated statistics from Evo downhole objects using the MCP data analysis utilities (shared with the tools) and stores them in a MongoDB collection. 

**Objectives**: 
- Test a basic implementation of the Evo MCP utilities > mongo DB integration
- Experiment with the calculated statistics to determine what is useful
- Assess performance on querying those statistics over a number of objects

**Steps:**
- Connects to Evo platform via hijacked OAuth token 
- Calculates interval statistics (length-weighted mean, accumulation, etc.)
- Calculates per-hole statistics
- Stores results with timestamps in MongoDB for tracking

**Prerequisites:**
- MongoDB running locally 
- Evo MCP configured with valid credentials in `.env`
- `pymongo` installed

#### Setup

In [1]:
import sys
import pandas as pd
import json
from pathlib import Path
from uuid import UUID

# Add src directory to path for imports
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Add notebooks directory to path for supporting_scripts
notebooks_path = Path.cwd()
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

# MongoDB utilities
from supporting_scripts.mongo_utils import (
    connect_to_mongodb,
    estimate_doc_size,
    build_gap_summary,
    prepare_collection_documents,
    create_hierarchy_indexes,
    get_ancestor_chain,
    get_children,
    build_object_uri,
    extract_referenced_uris,
    find_high_grade_objects,
    get_top_objects_by_grade,
    MONGO_DOC_LIMIT,
    SAFE_DOC_LIMIT,
)

# Benchmarking utilities
from supporting_scripts.stats_benchmark import (
    profile_query,
    get_explain_stats,
)

# Data loading utilities
from supporting_scripts.data_loading import download_all_interval_tables
from evo_mcp.utils.evo_data_utils import discover_objects, load_downhole_object

# Evo MCP utilities
from evo_mcp.context import evo_context, ensure_initialized
from evo_mcp.utils.data_analysis_utils import (
    calculate_interval_statistics,
    calculate_statistics_by_hole,
    calculate_categorical_statistics,
    analyze_gaps,
    calculate_multi_grade_statistics,
)

#### MongoDB onfiguration

Define the MongoDB connection settings and Evo workspace/object parameters.

In [2]:
# Load MongoDB connection parameters from config file

config_path = Path.cwd() / "mongo_config.json"
if config_path.exists():
    with open(config_path, 'r') as f:
        mongo_config = json.load(f)
    
    # Check if Atlas credentials are provided
    if "username" in mongo_config and "password" in mongo_config and "cluster_url" in mongo_config:
        protocol = mongo_config.get("protocol", "mongodb+srv")
        username = mongo_config["username"]
        password = mongo_config["password"]
        cluster_url = mongo_config["cluster_url"]
        MONGO_URI = f"{protocol}://{username}:{password}@{cluster_url}"
        print(f"Using MongoDB Atlas ({protocol}): {cluster_url.split('/')[0]}")
    else:
        # Fall back to local MongoDB
        protocol = mongo_config.get("protocol", "mongodb")
        host = mongo_config.get("host", "localhost")
        port = mongo_config.get("port", 27017)
        MONGO_URI = f"{protocol}://{host}:{port}/"
        print(f"Using local MongoDB: {host}:{port}")
    
    MONGO_DB_NAME = mongo_config.get("database", "evo")
    MONGO_COLLECTION_NAME = mongo_config.get("collection", "interval_statistics")
else:
    print(f"Config file not found: {config_path}")
    MONGO_URI = "mongodb://localhost:27017/"
    MONGO_DB_NAME = "evo"
    MONGO_COLLECTION_NAME = "interval_statistics"

WORKSPACE_ID = "01c54ab3-0b97-4b36-8e72-686e65a906ed"

# Optional: specific version (leave empty for latest)
VERSION = ""

# If False, skip objects that already have documents in the collection
OVERWRITE_STATS = True

# Evo web base URL for building object URIs
EVO_WEB_BASE_URL = "https://evo.integration.seequent.com"

print(f"MongoDB: {MONGO_URI.split('@')[-1] if '@' in MONGO_URI else MONGO_URI}")
print(f"Database: {MONGO_DB_NAME}.{MONGO_COLLECTION_NAME}")
print(f"Overwrite: {OVERWRITE_STATS}")
print(f"Workspace: {WORKSPACE_ID}")
print(f"Evo Web:   {EVO_WEB_BASE_URL}")

Using MongoDB Atlas (mongodb+srv): sq-labs-clickops-david.9lxzxzh.mongodb.net
MongoDB: sq-labs-clickops-david.9lxzxzh.mongodb.net/labs-api?retryWrites=true&w=majority&appName=sq-labs-clickops-david
Database: evo.interval_statistics
Overwrite: True
Workspace: 01c54ab3-0b97-4b36-8e72-686e65a906ed
Evo Web:   https://evo.integration.seequent.com


## 3. Connect to MongoDB

Establish connection to MongoDB and create/access the target collection.

Make sure you've built the docker image for the server and that is running before executing this cell.The docker command to run the server is:

```bash
docker run -d -p 27017:27017 --name mongodb mongo:latest
```

In [3]:
mongo_client, mongo_db, stats_collection = connect_to_mongodb(MONGO_URI, MONGO_DB_NAME, MONGO_COLLECTION_NAME)

✓ Connected to MongoDB: evo.interval_statistics


In [4]:
# Query existing data in the collection
doc_count = stats_collection.count_documents({})
print(f"Collection '{MONGO_COLLECTION_NAME}' has {doc_count} document(s)\n")

if doc_count > 0:
    # Distinct objects already stored
    stored_objects = stats_collection.distinct("object_id")
    print(f"Objects stored: {len(stored_objects)}")

    # Breakdown by object
    pipeline = [
        {"$group": {
            "_id": {"object_id": "$object_id", "object_name": "$object_name", "object_type": "$object_type"},
            "doc_count": {"$sum": 1},
            "collections": {"$addToSet": "$collection_name"},
            "latest": {"$max": "$timestamp"},
        }},
        {"$sort": {"_id.object_name": 1}},
    ]
    for row in stats_collection.aggregate(pipeline):
        info = row["_id"]
        print(f"\n  {info.get('object_name', '?')} ({info.get('object_type', '?')})")
        print(f"    ID: {info.get('object_id', '?')}")
        print(f"    Documents: {row['doc_count']}, Collections: {row['collections']}")
        print(f"    Latest: {row['latest']}")
else:
    print("Collection is empty — no data has been inserted yet.")

Collection 'interval_statistics' has 10 document(s)

Objects stored: 3

  ? (?)
    ID: ?
    Documents: 2, Collections: []
    Latest: 2026-02-18T21:47:20.567789+00:00

  Maia Drillholes (downhole-collection)
    ID: 0286ea01-1a2c-41a8-81b8-fca7df38feac
    Documents: 3, Collections: ['assay', 'geology']
    Latest: 2026-02-18 21:41:18.052000

  Maia Geology (Desurveyed) (downhole-intervals)
    ID: ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e
    Documents: 2, Collections: ['intervals']
    Latest: 2026-02-18 21:43:58.171000

  Wolfpass Drillhole Data (downhole-collection)
    ID: 7999f234-1e93-4cfe-8f43-15d244c85680
    Documents: 3, Collections: ['lithology', 'assay']
    Latest: 2026-02-18 21:42:51.725000


## 4. Initialize Evo Connection

Initialize the Evo SDK context and authenticate via OAuth.

In [5]:
# Initialize Evo SDK connection (will trigger OAuth flow if needed)
await ensure_initialized()
print("Evo SDK initialized and authenticated")

Evo SDK initialized and authenticated


## 5. Load Object and Inspect Collections

Download the Evo object and inspect available collections/attributes.

In [20]:
# Discover all objects in the workspace, filtered to downhole types
all_objects = await discover_objects(
    WORKSPACE_ID,
    object_types=["downhole-collection", "downhole-intervals"],
)

print(f"Found {len(all_objects)} object(s) matching types ['downhole-collection', 'downhole-intervals']")
for o in all_objects:
    print(f"  {o['name']} ({o['schema_id']}) — {o['id']}")

Found 3 object(s) matching types ['downhole-collection', 'downhole-intervals']
  maia.json (downhole-collection) — 0286ea01-1a2c-41a8-81b8-fca7df38feac
  wolfpass.json (downhole-collection) — 7999f234-1e93-4cfe-8f43-15d244c85680
  maia_geology_desurveyed.json (downhole-intervals) — ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e


## 5.1 Intialize ADK summarization agent

In [21]:
from google.adk.agents import LlmAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmResponse
from google.genai import Client as GenaiClient

GOOGLE_GENAI_USE_VERTEXAI=1
GOOGLE_CLOUD_PROJECT='sq-sbx-projs-evoopenmcp-001'
GOOGLE_CLOUD_LOCATION='global'
EVO_AGENT_MODEL='gemini-3-flash-preview'

# --- Embedding configuration ---
EMBEDDING_MODEL = "text-embedding-004"
EMBEDDING_LOCATION = "us-central1"  # Embedding models require a regional endpoint

# Create a dedicated client for embeddings (separate location from ADK agent)
embed_client = GenaiClient(
    vertexai=True,
    project=GOOGLE_CLOUD_PROJECT,
    location=EMBEDDING_LOCATION,
)

def get_embedding(text: str) -> list[float]:
    """Compute a text embedding using Vertex AI text-embedding-004.
    
    Uses task_type RETRIEVAL_DOCUMENT since these embeddings represent
    stored documents that will be searched against later.
    """
    response = embed_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config={"task_type": "RETRIEVAL_DOCUMENT"},
    )
    return response.embeddings[0].values

print(f"Embedding client ready: {EMBEDDING_MODEL} @ {EMBEDDING_LOCATION}")

instruction_str = """\
You are a geoscience data summarization agent. You produce detailed, comprehensive \
summaries of structured statistics derived from geoscience objects.

<role>
- Expert geoscientist and data analyst
- Specialist in drillhole, point, mesh, and geological model data
- Skilled at interpreting per-hole and overall statistics to identify standout results
</role>

<task>
Analyze the JSON-formatted statistics provided by the user. The data includes both \
overall (aggregate) statistics and per-hole (by_hole) statistics for each attribute. \
You must incorporate ALL of this information into a thorough summary.
</task>

<rules>
- Identify the data type (drillholes, points, meshes, geological models, etc.)
- Describe the dataset's purpose and content
- Infer geological context where reasonable (mineral exploration, structural geology, resource modelling)
- Report ALL overall statistics for every attribute (length_weighted_mean, accumulation, max, min, count, etc.)
- Analyse the per-hole (by_hole) data for each attribute:
  - List the **top 10 holes by length-weighted mean** (hole_id and value)
  - List the **top 5 holes by max grade** (hole_id and value)
  - Note holes with the longest total sampled length
  - Note holes with unusually high accumulation (grade × metres)
  - Report the range and spread across holes (best vs worst)
- For categorical attributes, report unique counts, top values, and per-hole distribution highlights
- Report gap analysis: total gap count, affected holes, gap lengths
- Use appropriate geological terminology but avoid unnecessary jargon
- Do NOT speculate beyond what the statistics reasonably support
- Do NOT open with filler phrases like "Based on the provided metadata" or "As an AI language model"
- Be thorough — include every statistic present in the data
</rules>

<output_format>
Structure the summary with these sections:

1. **Data overview** — object type, object name, collection name, attribute count, hole count
2. **Geological context** — inferred setting, commodity, or domain
3. **Overall statistics per attribute** — table or list of all aggregate values
4. **Per-hole highlights** — for each numeric attribute:
   - Top 10 holes by length-weighted mean (hole_id: value)
   - Top 5 holes by maximum grade (hole_id: value)
   - Holes with highest accumulation
   - Range across holes (min to max LWM)
5. **Categorical attributes** — unique values, dominant categories, per-hole patterns
6. **Gap analysis** — total gaps, affected holes, gap length statistics
7. **Data quality notes** — missing values, coverage, anomalies
</output_format>

"""

root_agent = LlmAgent(
    model=EVO_AGENT_MODEL,
    name='stats_summarization_agent',
    description='Summarizes geoscience object statistics into concise, human-readable geological narratives.',
    instruction=instruction_str
)


import json
from google.adk.runners import InMemoryRunner
from google.genai import types
# Run the agent — pass stats as the user message
APP_NAME = "stats_summarization_agent"
runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id="notebook_user"
)

Embedding client ready: text-embedding-004 @ us-central1


## 6. Process All Objects → MongoDB → Agent Summary

For each discovered object:
1. **Load** the object and inspect its interval collections
2. **Download** all interval tables, classifying columns as numeric vs categorical
3. **Calculate** statistics (overall + per-hole) and gap analysis for every numeric attribute
4. **Prepare** MongoDB documents (splitting large docs to stay under the 14 MB safe limit)
5. **Insert** into MongoDB
6. **Summarize** each document via the ADK agent and store the summary back in MongoDB

In [ ]:
import time

total_docs_inserted = 0
skipped_objects = []
failed_objects = []
inserted_object_ids = []  # Cache IDs of all successfully processed objects
object_timings = []  # Collect per-object timing breakdowns

# Pre-fetch existing object IDs from the collection
existing_object_ids = set(stats_collection.distinct("object_id")) if not OVERWRITE_STATS else set()

cell_start = time.perf_counter()

for obj_idx, obj_meta in enumerate(all_objects, 1):
    obj_id = obj_meta["id"]
    obj_name = obj_meta["name"]
    
    print(f"\n{'#'*70}")
    print(f"[{obj_idx}/{len(all_objects)}] {obj_name} ({obj_id})")
    print(f"{'#'*70}")
    
    # Skip if already in the database and not overwriting
    if obj_id in existing_object_ids:
        print(f"  Skipped (already in database)")
        skipped_objects.append(obj_name)
        inserted_object_ids.append(obj_id)  # Still track — data exists in DB
        continue
    
    obj_start = time.perf_counter()
    timings = {"object": obj_name}
    
    try:
        # --- 1. Load object ---
        t1 = time.perf_counter()
        obj, obj_dict, object_name, object_type, collections_info = await load_downhole_object(
            WORKSPACE_ID, obj_id, VERSION
        )
        timings["1_load"] = time.perf_counter() - t1
        # print(obj_dict)
        print(f"  Type: {object_type}, {len(collections_info)} interval table(s)")
        for coll in collections_info:
            attrs = coll.get("attributes", [])
            print(f"    {coll['name']} ({len(attrs)} attributes)")
        print(f"  Step 1 (load object): {timings['1_load']:.1f}s")

        # --- 2. Download interval tables ---
        t2 = time.perf_counter()
        collection_data = await download_all_interval_tables(obj, object_type, collections_info)
        timings["2_download"] = time.perf_counter() - t2
        
        for coll_name, coll_data in collection_data.items():
            df = coll_data["df"]
            elapsed = coll_data["elapsed"]
            print(f"  {coll_name}: {len(df):,} intervals, "
                  f"{len(coll_data['numeric_cols'])} numeric / {len(coll_data['categorical_cols'])} categorical cols "
                  f"({elapsed:.1f}s)")
        print(f"  Step 2 (download): {timings['2_download']:.1f}s")

        # --- 3. Calculate statistics ---
        t3 = time.perf_counter()
        all_collection_stats = {}
        
        for coll_name, coll_data in collection_data.items():
            df = coll_data["df"]
            numeric_cols = coll_data["numeric_cols"]
            categorical_cols = coll_data["categorical_cols"]
            attribute_stats = {}
            
            # 3a. Numeric attribute statistics (LWM, accumulation, per-hole, etc.)
            for grade_col in numeric_cols:
                try:
                    overall = calculate_interval_statistics(df, grade_col)
                except (ValueError, ZeroDivisionError):
                    continue
                
                hole_stats_df = calculate_statistics_by_hole(df, grade_col)
                attribute_stats[grade_col] = {
                    "overall": overall,
                    "hole_count": len(hole_stats_df),
                    "by_hole": hole_stats_df.to_dict(orient="records"),
                }
            
            # 3b. Categorical attribute statistics (value counts, per-hole breakdown)
            categorical_stats = {}
            for cat_col in categorical_cols:
                try:
                    cat_stats = calculate_categorical_statistics(df, cat_col)
                    categorical_stats[cat_col] = cat_stats
                except (ValueError, Exception) as e:
                    print(f"    Warning: skipped categorical stat for '{cat_col}': {e}")
            
            gap_analysis = analyze_gaps(df)
            all_collection_stats[coll_name] = {
                "attributes": attribute_stats,
                "categorical": categorical_stats,
                "gap_analysis": gap_analysis,
            }
        
        timings["3_stats"] = time.perf_counter() - t3
        total_attrs = sum(len(v["attributes"]) for v in all_collection_stats.values())
        total_cats = sum(len(v["categorical"]) for v in all_collection_stats.values())
        print(f"  {total_attrs} numeric + {total_cats} categorical statistics across {len(all_collection_stats)} table(s)")
        print(f"  Step 3 (calculate stats): {timings['3_stats']:.1f}s")

        # --- 4. Prepare MongoDB documents ---
        t4 = time.perf_counter()
        all_documents = []
        for coll_name, coll_stats in all_collection_stats.items():
            docs = prepare_collection_documents(
                workspace_id=WORKSPACE_ID,
                object_id=obj_id,
                object_name=object_name,
                object_type=object_type,
                collection_name=coll_name,
                attribute_stats=coll_stats["attributes"],
                gap_analysis=coll_stats["gap_analysis"],
                categorical_stats=coll_stats["categorical"],
            )
            all_documents.extend(docs)
        
        timings["4_prepare"] = time.perf_counter() - t4
        total_size_mb = sum(d["metadata"].get("doc_size_bytes", 0) for d in all_documents) / 1024 / 1024
        print(f"  {len(all_documents)} document(s), {total_size_mb:.2f} MB total")
        print(f"  Step 4 (prepare docs): {timings['4_prepare']:.1f}s")

        # --- 5. Insert into MongoDB ---
        t5 = time.perf_counter()
        if all_documents:
            result = stats_collection.insert_many(all_documents)
            total_docs_inserted += len(result.inserted_ids)
            inserted_object_ids.append(obj_id)
            print(f"  Inserted {len(result.inserted_ids)} document(s)")
        else:
            print(f"  No documents to insert")
        timings["5_insert"] = time.perf_counter() - t5
        print(f"  Step 5 (MongoDB insert): {timings['5_insert']:.1f}s")

        # --- 6. Summarize + embed each document with the ADK agent ---
        t6 = time.perf_counter()
        for doc_idx, doc in enumerate(all_documents, 1):
            doc_start = time.perf_counter()
            
            # Strip MongoDB _id (not JSON-serializable) for the prompt
            doc_for_prompt = {k: v for k, v in doc.items() if k != "_id"}
            doc_json = json.dumps(doc_for_prompt, indent=2, default=str)

            # Create a fresh session per document
            doc_session = await runner.session_service.create_session(
                app_name=APP_NAME, user_id="notebook_user"
            )

            # 6a. Generate agent summary
            summary_parts = []
            agent_response = runner.run_async(
                user_id="notebook_user",
                session_id=doc_session.id,
                new_message=types.Content(
                    role="user",
                    parts=[types.Part(text=(
                        "Produce a detailed summary of the following statistics. "
                        "Include ALL overall values and analyse the per-hole (by_hole) data — "
                        "rank the top 10 holes by length-weighted mean, top 5 by max grade, "
                        "and note holes with highest accumulation and longest sampled length.\n\n"
                        f"{doc_json}"
                    ))]
                ),
            )
            async for event in agent_response:
                if event.content and event.content.parts:
                    for part in event.content.parts:
                        if part.text:
                            summary_parts.append(part.text)

            summary_text = "".join(summary_parts)
            t_summ = time.perf_counter() - doc_start

            # 6b. Compute text embedding for the summary
            t_embed_start = time.perf_counter()
            summary_embedding = get_embedding(summary_text)
            t_embed = time.perf_counter() - t_embed_start

            # Update the MongoDB document with agent summary + embedding
            if doc.get("_id"):
                comp_uri = build_object_uri(
                    EVO_WEB_BASE_URL, str(evo_context.org_id),
                    WORKSPACE_ID, obj_id,
                )
                stats_collection.update_one(
                    {"_id": doc["_id"]},
                    {"$set": {
                        "agent_summary": summary_text,
                        "agent_summary_embedding": summary_embedding,
                        "hierarchy_level": "component",
                        "referenced_uris": [comp_uri],
                    }}
                )

            doc_elapsed = time.perf_counter() - doc_start
            coll_name = doc.get("collection_name", "?")
            doc_type = doc.get("doc_type", "?")
            prompt_kb = len(doc_json) / 1024
            print(f"    [{doc_idx}/{len(all_documents)}] {coll_name} [{doc_type}] — "
                  f"summary ({len(summary_text)} chars, {t_summ:.1f}s), "
                  f"embed ({len(summary_embedding)}d, {t_embed:.1f}s), "
                  f"prompt {prompt_kb:.0f} KB, total {doc_elapsed:.1f}s")
        
        timings["6_agent"] = time.perf_counter() - t6
        timings["6_agent_per_doc"] = timings["6_agent"] / max(len(all_documents), 1)
        print(f"  Step 6 (agent + embed): {timings['6_agent']:.1f}s total, "
              f"{timings['6_agent_per_doc']:.1f}s/doc avg ({len(all_documents)} docs)")

        timings["total"] = time.perf_counter() - obj_start
        object_timings.append(timings)
        print(f"\n  Object total: {timings['total']:.1f}s "
              f"[load={timings['1_load']:.1f} | dl={timings['2_download']:.1f} | "
              f"stats={timings['3_stats']:.1f} | prep={timings['4_prepare']:.1f} | "
              f"insert={timings['5_insert']:.1f} | agent+embed={timings['6_agent']:.1f}]")
    
    except Exception as e:
        failed_objects.append({"name": obj_name, "id": obj_id, "error": str(e)})
        print(f"  FAILED: {e}")

# --- Summary ---
cell_elapsed = time.perf_counter() - cell_start
print(f"\n{'='*70}")
print(f"Done: {len(all_objects)} objects, {total_docs_inserted} documents inserted, "
      f"{len(skipped_objects)} skipped, {len(failed_objects)} failed")
print(f"Total cell time: {cell_elapsed:.1f}s")
print(f"Cached {len(inserted_object_ids)} object ID(s) for downstream queries")

if object_timings:
    print(f"\n{'─'*70}")
    print(f"Timing breakdown per object:")
    print(f"  {'Object':<30} {'Load':>6} {'DL':>6} {'Stats':>6} {'Prep':>6} {'Insert':>6} {'Ag+Em':>6} {'Total':>7}")
    for t in object_timings:
        print(f"  {t['object'][:30]:<30} {t['1_load']:>5.1f}s {t['2_download']:>5.1f}s "
              f"{t['3_stats']:>5.1f}s {t['4_prepare']:>5.1f}s {t['5_insert']:>5.1f}s "
              f"{t['6_agent']:>5.1f}s {t['total']:>6.1f}s")

if skipped_objects:
    print(f"\n{len(skipped_objects)} skipped (already in database):")
    for name in skipped_objects:
        print(f"  {name}")
if failed_objects:
    print(f"\n{len(failed_objects)} failed:")
    for f in failed_objects:
        print(f"  {f['name']}: {f['error']}")
if not OVERWRITE_STATS and skipped_objects:
    print(f"\nSet OVERWRITE_STATS = True to re-process skipped objects.")


######################################################################
[1/3] maia.json (0286ea01-1a2c-41a8-81b8-fca7df38feac)
######################################################################
{'schema': '/objects/downhole-collection/1.3.0/downhole-collection.schema.json', 'uuid': UUID('0286ea01-1a2c-41a8-81b8-fca7df38feac'), 'name': 'Maia Drillholes', 'description': 'Maia gold drilling dataset with collar, survey, assay (Au), and geology data', 'tags': {}, 'bounding_box': {'min_x': 543999.23, 'max_x': 544500.94, 'min_y': 4317099.17, 'max_y': 4317501.1, 'min_z': 693.874, 'max_z': 729.739}, 'coordinate_reference_system': 'unspecified', 'location': {'coordinates': {'data': 'c561fcb491c2bc65af2a53849a4a3dae4f5a5882dec6e20fd3f09a48babee570', 'length': 14, 'width': 3, 'data_type': 'float64'}, 'distances': {'data': '95837752576cc3418ca59c1ae22a4fdd8f794831556ee6e7b1a475d84880ec3a', 'length': 14, 'width': 3, 'data_type': 'float64'}, 'holes': {'data': '64b470965edbcdcea3ed56fec6309097d265

## 6.1 Generate Hierarchical Summaries

After processing all data objects, generate summary documents at every level of the hierarchy. Each level rolls up from the one below, and each document describes a different **entity** (the organisation, a workspace, a data object, or a component):

ORGANISATION: portfolio-wide overview across all workspaces
WORKSPACE: comparison across data objects within one workspace
OBJECT: cross-component comparison within one data object
COMPONENT: detailed per-attribute, per-hole statistics (existing docs)

Each entity level has its own agent summary + embedding, tuned so that embedding similarity naturally routes queries to the correct hierarchy level based on question scope.

In [ ]:
import re
import time
from datetime import datetime, timezone

print("Generating hierarchical summaries...\n")

# ══════════════════════════════════════════════════════════════════════════
# Agent instructions for each hierarchy level
# ══════════════════════════════════════════════════════════════════════════

object_instruction = """\
You are a geoscience data summarization agent producing an **OBJECT-LEVEL** summary.

<hierarchy>
This summary sits at the OBJECT level in a four-tier hierarchy:
  ORGANISATION → WORKSPACE → OBJECT → COMPONENT
Each level describes a different entity: the organisation portfolio, a workspace, \
a data object, or an interval table (component).
You are summarizing across ALL interval tables (components) within a single \
data object. A user searching at this level wants to understand the full \
picture of one data object — what interval tables it has, how attributes compare \
across tables, and overall data quality.
</hierarchy>

<task>
You are given the individual component-level summaries for every interval table \
in this data object. Synthesize them into a single cohesive entity overview.
</task>

<rules>
- Open with: "OBJECT SUMMARY — [object_name] ([object_type])"
- State the total number of interval tables (components), their names, and attribute counts
- For each component, note: hole count, key numeric attributes, notable grades, data type
- Compare components: which has the most attributes? highest grades? most holes?
- Highlight the most important grade attributes across all components (e.g. Au, Cu)
- Report overall hole count and approximate total intervals across all tables
- Note gap analysis highlights (total gaps, affected holes)
- Note data quality observations that span multiple components
- Do NOT repeat full per-hole rankings from each component — synthesize key highlights
- Do NOT open with filler phrases like "Based on the provided data"
</rules>

<output_format>
1. **Entity overview** — data object name, type, total components, total hole count
2. **Components inventory** — list each interval table with attribute count, hole count, key stats
3. **Cross-component comparison** — grades, data volume, notable differences
4. **Key attribute highlights** — top grades, LWM, accumulations across all tables
5. **Gap analysis summary** — total gaps, affected holes across all tables
6. **Data quality notes** — missing data, coverage, anomalies
</output_format>
"""

workspace_instruction = """\
You are a geoscience data summarization agent producing a **WORKSPACE-LEVEL** summary.

<hierarchy>
This summary sits at the WORKSPACE level in a four-tier hierarchy:
  ORGANISATION → WORKSPACE → OBJECT → COMPONENT
Each level describes a different entity: the organisation portfolio, a workspace, \
a data object, or an interval table (component).
You are summarizing across ALL data objects within a single workspace.
A user searching at this level wants to understand the full scope of drilling \
data in this workspace — what data objects exist, how they compare, and what \
patterns emerge across the entire workspace.
</hierarchy>

<task>
You are given the individual object-level summaries for every data object \
in this workspace. Synthesize them into a single cohesive entity overview.
</task>

<rules>
- Open with: "WORKSPACE SUMMARY — [workspace_id]"
- State the total number of data objects, their types, and component names
- For each data object, note: component count, hole count, key attributes, notable grades
- Compare data objects: which has the most holes? highest grades? longest intervals?
- Identify workspace-wide patterns: common attributes across data objects, grade ranges
- Highlight standout data objects (highest LWM, richest holes, most data)
- Report total data scope: approximate total holes, total intervals, total attributes
- Note data quality observations that span multiple data objects
- Do NOT repeat the full detail from each object-level summary — synthesize and compare
- Do NOT open with filler phrases like "Based on the provided data"
</rules>

<output_format>
1. **Entity overview** — workspace ID, data object count, data object types
2. **Data objects inventory** — list each data object with type, components, hole count, key attributes
3. **Cross-entity comparison** — grades, hole counts, data volume, notable differences
4. **Workspace-wide patterns** — common attributes, grade ranges, geological themes
5. **Standout highlights** — best grades, longest holes, richest accumulations
6. **Data quality overview** — gaps, missing data patterns across data objects
</output_format>
"""

org_instruction = """\
You are a geoscience data summarization agent producing an **ORGANISATION-LEVEL** summary.

<hierarchy>
This summary sits at the ORGANISATION level — the highest tier:
  ORGANISATION → WORKSPACE → OBJECT → COMPONENT
Each level describes a different entity: the organisation portfolio, a workspace, \
a data object, or an interval table (component).
You are summarizing across ALL workspaces in the organisation's database.
A user searching at this level wants a bird's-eye view of all drilling data \
available — which workspaces exist, what commodities are covered, and how \
the overall data portfolio looks.
</hierarchy>

<task>
You are given workspace-level summaries for every workspace that has been \
processed. Synthesize them into a single organisation-wide entity overview.
</task>

<rules>
- Open with: "ORGANISATION SUMMARY — Drilling Data Portfolio"
- State the total number of workspaces and total data objects across all workspaces
- For each workspace, note: data object count, key commodities/attributes, data scope
- Compare workspaces: which has the most data? highest grade potential?
- Identify organisation-wide patterns: commodities, geological settings, data coverage
- Report total data portfolio: workspaces, data objects, approximate holes and intervals
- Note cross-workspace data quality themes
- Do NOT repeat workspace-level detail verbatim — synthesize at a higher level
- Do NOT open with filler phrases like "Based on the provided data"
</rules>

<output_format>
1. **Entity overview** — total workspaces, total data objects, data scope
2. **Workspace inventory** — list each workspace with key stats
3. **Cross-workspace comparison** — size, commodities, grade potential
4. **Portfolio patterns** — commodities covered, geological contexts, data maturity
5. **Strategic highlights** — highest-value datasets, exploration priorities
6. **Data quality portfolio** — coverage and quality themes across workspaces
</output_format>
"""


# ══════════════════════════════════════════════════════════════════════════
# A. Object-level summaries (one entity per data object, aggregating its components)
# ══════════════════════════════════════════════════════════════════════════

t_obj_start = time.perf_counter()

# Find distinct objects in this workspace that have component-level summaries
obj_pipeline = [
    {"$match": {
        "workspace_id": WORKSPACE_ID,
        "hierarchy_level": "component",
        "agent_summary": {"$exists": True, "$ne": ""},
    }},
    {"$group": {
        "_id": {"object_id": "$object_id", "object_name": "$object_name", "object_type": "$object_type"},
        "collections": {"$addToSet": "$collection_name"},
    }},
    {"$sort": {"_id.object_name": 1}},
]
distinct_objects = list(stats_collection.aggregate(obj_pipeline))
print(f"Found {len(distinct_objects)} distinct data object(s) to summarize at OBJECT level\n")

# Create object-level summarization agent
obj_summary_agent = LlmAgent(
    model=EVO_AGENT_MODEL,
    name="object_summary_agent",
    description="Produces object-level summaries from component-level summaries.",
    instruction=object_instruction,
)
obj_runner = InMemoryRunner(agent=obj_summary_agent, app_name="object_summary_agent")

for obj_info in distinct_objects:
    obj_id = obj_info["_id"]["object_id"]
    obj_name = obj_info["_id"]["object_name"]
    obj_type = obj_info["_id"]["object_type"]
    collections = obj_info["collections"]
    
    print(f"  {obj_name} ({obj_type}) — {len(collections)} component(s): {', '.join(collections)}")
    
    # Gather component-level summaries for this object
    coll_docs = list(stats_collection.find(
        {
            "object_id": obj_id,
            "hierarchy_level": "component",
            "agent_summary": {"$exists": True, "$ne": ""},
        },
        {
            "object_name": 1, "object_type": 1, "collection_name": 1,
            "doc_type": 1, "data_type": 1, "agent_summary": 1,
            "gap_analysis": 1,
        },
    ))
    
    obj_context = []
    for doc in coll_docs:
        obj_context.append({
            "collection_name": doc.get("collection_name", "?"),
            "doc_type": doc.get("doc_type", "?"),
            "data_type": doc.get("data_type", "?"),
            "gap_analysis": doc.get("gap_analysis", {}),
            "agent_summary": doc.get("agent_summary", ""),
        })
    
    obj_context_json = json.dumps({
        "object_name": obj_name,
        "object_type": obj_type,
        "object_id": obj_id,
        "collection_count": len(collections),
        "collections": obj_context,
    }, indent=2, default=str)
    
    # Generate object summary
    obj_session = await obj_runner.session_service.create_session(
        app_name="object_summary_agent", user_id="notebook_user"
    )
    obj_parts = []
    obj_response = obj_runner.run_async(
        user_id="notebook_user",
        session_id=obj_session.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=(
                "Produce an object-level summary for the following data object. "
                "Synthesize the individual component summaries into a cohesive entity overview.\n\n"
                f"{obj_context_json}"
            ))],
        ),
    )
    async for event in obj_response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    obj_parts.append(part.text)
    
    obj_summary_text = "".join(obj_parts)
    obj_summary_embedding = get_embedding(obj_summary_text)
    
    # Build the self-referencing URI for this object
    obj_uri = build_object_uri(
        EVO_WEB_BASE_URL, str(evo_context.org_id), WORKSPACE_ID, obj_id,
    )

    # Upsert object summary document (stable _id for parent_id references)
    obj_filter = {"hierarchy_level": "object", "object_id": obj_id}
    obj_doc = {
        "hierarchy_level": "object",
        "workspace_id": WORKSPACE_ID,
        "object_id": obj_id,
        "object_name": obj_name,
        "object_type": obj_type,
        "doc_type": "object_summary",
        "collection_count": len(collections),
        "collections": sorted(collections),
        "agent_summary": obj_summary_text,
        "agent_summary_embedding": obj_summary_embedding,
        "referenced_uris": [obj_uri],
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    obj_result = stats_collection.update_one(obj_filter, {"$set": obj_doc}, upsert=True)
    obj_mongo_id = obj_result.upserted_id or stats_collection.find_one(obj_filter, {"_id": 1})["_id"]

    # Backfill parent_id on component children of this object
    stats_collection.update_many(
        {"hierarchy_level": "component", "object_id": obj_id},
        {"$set": {"parent_id": obj_mongo_id}},
    )

    print(f"    → {len(obj_summary_text)} chars, {len(obj_summary_embedding)}d embedding, _id={obj_mongo_id}")

t_obj_elapsed = time.perf_counter() - t_obj_start
print(f"\n  Object-level summaries: {t_obj_elapsed:.1f}s for {len(distinct_objects)} object(s)")


# ══════════════════════════════════════════════════════════════════════════
# B. Workspace-level summary (one entity aggregating all object-level summaries)
# ══════════════════════════════════════════════════════════════════════════

t_ws_start = time.perf_counter()

# Gather object-level summaries for this workspace
workspace_object_docs = list(stats_collection.find(
    {
        "workspace_id": WORKSPACE_ID,
        "agent_summary": {"$exists": True, "$ne": ""},
        "hierarchy_level": "object",
    },
    {
        "object_name": 1, "object_type": 1, "object_id": 1,
        "collection_count": 1, "collections": 1, "doc_type": 1,
        "agent_summary": 1,
    },
))

print(f"\nFound {len(workspace_object_docs)} object-level summaries for workspace {WORKSPACE_ID}")

# Build objects_map for URI extraction: {object_name: object_id}
ws_objects_map = {
    doc["object_name"]: doc["object_id"]
    for doc in workspace_object_docs
    if "object_name" in doc and "object_id" in doc
}

ws_objects_context = []
for doc in workspace_object_docs:
    ws_objects_context.append({
        "object_name": doc.get("object_name", "?"),
        "object_type": doc.get("object_type", "?"),
        "collection_count": doc.get("collection_count", 0),
        "collections": doc.get("collections", []),
        "agent_summary": doc.get("agent_summary", ""),
    })

ws_context_json = json.dumps({
    "workspace_id": WORKSPACE_ID,
    "object_count": len(ws_objects_context),
    "objects": ws_objects_context,
}, indent=2, default=str)

# Create workspace-level summarization agent
ws_summary_agent = LlmAgent(
    model=EVO_AGENT_MODEL,
    name="workspace_summary_agent",
    description="Produces workspace-level summaries from object-level summaries.",
    instruction=workspace_instruction,
)
ws_runner = InMemoryRunner(agent=ws_summary_agent, app_name="workspace_summary_agent")
ws_session = await ws_runner.session_service.create_session(
    app_name="workspace_summary_agent", user_id="notebook_user"
)

ws_parts = []
ws_response = ws_runner.run_async(
    user_id="notebook_user",
    session_id=ws_session.id,
    new_message=types.Content(
        role="user",
        parts=[types.Part(text=(
            "Produce a workspace-level summary for the following workspace. "
            "Synthesize the individual object summaries into a cohesive entity overview.\n\n"
            f"{ws_context_json}"
        ))],
    ),
)
async for event in ws_response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                ws_parts.append(part.text)

ws_summary_text = "".join(ws_parts)
ws_summary_embedding = get_embedding(ws_summary_text)
t_ws_summary = time.perf_counter() - t_ws_start

# Extract referenced URIs — only objects named in the workspace summary
ws_referenced_uris = extract_referenced_uris(
    ws_summary_text, ws_objects_map,
    EVO_WEB_BASE_URL, str(evo_context.org_id), WORKSPACE_ID,
)

ws_filter = {"hierarchy_level": "workspace", "workspace_id": WORKSPACE_ID}
ws_doc = {
    "hierarchy_level": "workspace",
    "workspace_id": WORKSPACE_ID,
    "doc_type": "workspace_summary",
    "object_count": len(ws_objects_context),
    "objects": sorted(d["object_name"] for d in ws_objects_context),
    "agent_summary": ws_summary_text,
    "agent_summary_embedding": ws_summary_embedding,
    "referenced_uris": ws_referenced_uris,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}
ws_result = stats_collection.update_one(ws_filter, {"$set": ws_doc}, upsert=True)
ws_mongo_id = ws_result.upserted_id or stats_collection.find_one(ws_filter, {"_id": 1})["_id"]

# Backfill parent_id on object children in this workspace
stats_collection.update_many(
    {"hierarchy_level": "object", "workspace_id": WORKSPACE_ID},
    {"$set": {"parent_id": ws_mongo_id}},
)

print(f"Workspace summary: {len(ws_summary_text)} chars, {t_ws_summary:.1f}s, _id={ws_mongo_id}")
print(f"  Referenced URIs: {len(ws_referenced_uris)}")
print(f"Preview:\n{ws_summary_text[:500]}...\n")


# ══════════════════════════════════════════════════════════════════════════
# C. Organisation-level summary (one entity aggregating all workspace-level summaries)
# ══════════════════════════════════════════════════════════════════════════

t_org_start = time.perf_counter()

all_ws_summaries = list(stats_collection.find(
    {"hierarchy_level": "workspace", "agent_summary": {"$exists": True}},
    {"workspace_id": 1, "object_count": 1, "objects": 1, "agent_summary": 1},
))

print(f"Found {len(all_ws_summaries)} workspace-level summaries for organisation roll-up")

# Build a cross-workspace objects_map for URI extraction
# Gather all object-level docs across all workspaces
all_org_object_docs = list(stats_collection.find(
    {"hierarchy_level": "object", "object_name": {"$exists": True}},
    {"object_name": 1, "object_id": 1, "workspace_id": 1},
))
# Map: {object_name: (object_id, workspace_id)} — last-write-wins for dupes
org_objects_lookup = {
    doc["object_name"]: (doc["object_id"], doc["workspace_id"])
    for doc in all_org_object_docs
    if "object_name" in doc and "object_id" in doc and "workspace_id" in doc
}

org_context = []
for ws in all_ws_summaries:
    org_context.append({
        "workspace_id": ws.get("workspace_id", "?"),
        "object_count": ws.get("object_count", 0),
        "objects": ws.get("objects", []),
        "workspace_summary": ws.get("agent_summary", ""),
    })

org_context_json = json.dumps({
    "workspace_count": len(all_ws_summaries),
    "total_objects": sum(ws.get("object_count", 0) for ws in all_ws_summaries),
    "workspaces": org_context,
}, indent=2, default=str)

org_summary_agent = LlmAgent(
    model=EVO_AGENT_MODEL,
    name="org_summary_agent",
    description="Produces organisation-level summaries from workspace-level summaries.",
    instruction=org_instruction,
)
org_runner = InMemoryRunner(agent=org_summary_agent, app_name="org_summary_agent")
org_session = await org_runner.session_service.create_session(
    app_name="org_summary_agent", user_id="notebook_user"
)

org_parts = []
org_response = org_runner.run_async(
    user_id="notebook_user",
    session_id=org_session.id,
    new_message=types.Content(
        role="user",
        parts=[types.Part(text=(
            "Produce an organisation-level summary of the entire drilling data portfolio. "
            "Synthesize the workspace summaries into a high-level entity overview.\n\n"
            f"{org_context_json}"
        ))],
    ),
)
async for event in org_response:
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                org_parts.append(part.text)

org_summary_text = "".join(org_parts)
org_summary_embedding = get_embedding(org_summary_text)
t_org_summary = time.perf_counter() - t_org_start

# Extract referenced URIs — scan org summary for object names across all workspaces
org_referenced_uris = []
for obj_name, (oid, ws_id) in org_objects_lookup.items():
    if re.search(re.escape(obj_name), org_summary_text, re.IGNORECASE):
        org_referenced_uris.append(
            build_object_uri(EVO_WEB_BASE_URL, str(evo_context.org_id), ws_id, oid)
        )
org_referenced_uris = list(dict.fromkeys(org_referenced_uris))  # deduplicate

org_filter = {"hierarchy_level": "organisation"}
org_doc = {
    "hierarchy_level": "organisation",
    "doc_type": "organisation_summary",
    "workspace_count": len(all_ws_summaries),
    "total_objects": sum(ws.get("object_count", 0) for ws in all_ws_summaries),
    "workspaces": [ws.get("workspace_id", "?") for ws in all_ws_summaries],
    "agent_summary": org_summary_text,
    "agent_summary_embedding": org_summary_embedding,
    "referenced_uris": org_referenced_uris,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}
org_result = stats_collection.update_one(org_filter, {"$set": org_doc}, upsert=True)
org_mongo_id = org_result.upserted_id or stats_collection.find_one(org_filter, {"_id": 1})["_id"]

# Backfill parent_id on workspace children
stats_collection.update_many(
    {"hierarchy_level": "workspace"},
    {"$set": {"parent_id": org_mongo_id}},
)

print(f"Organisation summary: {len(org_summary_text)} chars, {t_org_summary:.1f}s, _id={org_mongo_id}")
print(f"  Referenced URIs: {len(org_referenced_uris)}")
print(f"Preview:\n{org_summary_text[:500]}...\n")

# ── Final count ───────────────────────────────────────────────────────────
total_hierarchy = time.perf_counter() - t_obj_start
print(f"{'='*60}")
print(f"Hierarchical summaries complete in {total_hierarchy:.1f}s")

for level in ["component", "object", "workspace", "organisation"]:
    count = stats_collection.count_documents({"hierarchy_level": level})
    embed_count = stats_collection.count_documents({
        "hierarchy_level": level,
        "agent_summary_embedding": {"$exists": True},
    })
    uri_count = stats_collection.count_documents({
        "hierarchy_level": level,
        "referenced_uris": {"$exists": True, "$ne": []},
    })
    print(f"  {level}: {count} docs ({embed_count} with embeddings, {uri_count} with URIs)")

Generating hierarchical summaries...

Found 3 distinct data object(s) to summarize at OBJECT level

  Maia Drillholes (downhole-collection) — 2 component(s): assay, geology
    → 2746 chars, 768d embedding
  Maia Geology (Desurveyed) (downhole-intervals) — 1 component(s): intervals
    → 2297 chars, 768d embedding
  Wolfpass Drillhole Data (downhole-collection) — 2 component(s): assay, lithology
    → 3618 chars, 768d embedding

  Object-level summaries: 52.9s for 3 object(s)

Found 3 object-level summaries for workspace 01c54ab3-0b97-4b36-8e72-686e65a906ed
Workspace summary: 3976 chars, 20.0s
Preview:
WORKSPACE SUMMARY — 01c54ab3-0b97-4b36-8e72-686e65a906ed

### 1. Entity overview
*   **Workspace ID:** 01c54ab3-0b97-4b36-8e72-686e65a906ed
*   **Total Data Objects:** 3
*   **Data Object Types:** 2 × downhole-collection, 1 × downhole-intervals
*   **Total Scope:** 69 unique drillholes across two distinct project areas (Maia and Wolfpass), totaling approximately 22,374 metres of drilling

## 6.2 RAG Query Agent

Create an ADK agent that answers geoscience questions by:
1. Embedding the user's prompt using `text-embedding-004` (task type `RETRIEVAL_QUERY`)
2. Finding the most similar `agent_summary_embedding` vectors in MongoDB via cosine similarity — across **all hierarchy levels** (organisation, workspace, object, component)
3. Passing the retrieved summaries as context to a Gemini LLM to generate a grounded answer

The four-tier hierarchy ensures that broad questions ("what data is available?") naturally match organisation/workspace summaries, while specific questions ("what is the Au grade in the assays table?") match component-level summaries.

In [24]:
import numpy as np
from google.adk.tools import FunctionTool

# ---------------------------------------------------------------------------
# 1. Embedding helper for queries (RETRIEVAL_QUERY task type)
# ---------------------------------------------------------------------------

def get_query_embedding(text: str) -> list[float]:
    """Embed a user query for retrieval against stored document embeddings."""
    response = embed_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config={"task_type": "RETRIEVAL_QUERY"},
    )
    return response.embeddings[0].values


# ---------------------------------------------------------------------------
# 2. Vector search tool — cosine similarity in Python
#    (works with both local MongoDB and Atlas without a vector index)
# ---------------------------------------------------------------------------

def search_statistics(query: str, top_k: int = 1) -> list[dict]:
    """Search the MongoDB statistics collection for documents whose agent
    summaries are semantically similar to the query.

    The database contains summaries at four hierarchy levels:
      - organisation: portfolio-wide overview across all workspaces
      - workspace: comparison across data objects within a single workspace
      - object: cross-component comparison within one data object
      - component: detailed per-attribute, per-hole statistics for one interval table

    The embedding similarity will naturally route queries to the correct
    hierarchy level based on the question's scope.

    Args:
        query: Natural-language question about geoscience statistics.
        top_k: Number of top results to return (default 3).

    Returns:
        A list of dicts, each containing hierarchy_level, object_name,
        collection_name, doc_type, similarity score, and the agent_summary text.
    """
    top_k = 1
    query_vec = np.array(get_query_embedding(query), dtype=np.float32)

    cursor = stats_collection.find(
        {"agent_summary_embedding": {"$exists": True}},
        {
            "object_name": 1,
            "object_type": 1,
            "collection_name": 1,
            "doc_type": 1,
            "hierarchy_level": 1,
            "agent_summary": 1,
            "agent_summary_embedding": 1,
        },
    )

    scored = []
    for doc in cursor:
        doc_vec = np.array(doc["agent_summary_embedding"], dtype=np.float32)
        denom = np.linalg.norm(query_vec) * np.linalg.norm(doc_vec)
        similarity = float(np.dot(query_vec, doc_vec) / denom) if denom > 0 else 0.0
        scored.append({
            "hierarchy_level": doc.get("hierarchy_level", "object"),
            "object_name": doc.get("object_name", "?"),
            "object_type": doc.get("object_type", "?"),
            "collection_name": doc.get("collection_name", "?"),
            "doc_type": doc.get("doc_type", "?"),
            "similarity": round(similarity, 4),
            "agent_summary": doc.get("agent_summary", ""),
        })

    scored.sort(key=lambda x: x["similarity"], reverse=True)
    for s in scored:
        level = s['hierarchy_level'].upper()
        label = s['object_name'] if s['hierarchy_level'] == 'object' else s['hierarchy_level']
        print(f"  [{level}] {label} / {s['collection_name']} ({s['doc_type']}): {s['similarity']:.4f}")
    return scored[:top_k]


search_tool = FunctionTool(func=search_statistics)


# ---------------------------------------------------------------------------
# 3. Structured MongoDB query tool
#    Lets the agent query the actual statistics data by attribute, metric,
#    object, etc. — much better than embedding search for ranked/numeric Qs
# ---------------------------------------------------------------------------

def query_statistics(
    attribute: str = "",
    metric: str = "length_weighted_mean",
    min_value: float = None,
    object_name: str = "",
    top_n: int = 10,
    include_by_hole: bool = True,
) -> list[dict]:
    """Query the MongoDB statistics collection using structured filters.

    Use this tool when the user asks about specific attributes (e.g. Au, Cu),
    wants ranked lists (top holes, highest grades), or needs exact numeric
    values across objects. This is more precise than semantic search.

    The document schema has:
      - object_name, object_type, collection_name, doc_type, data_type
      - statistics[]: array where each element has:
          - attribute: column name (e.g. "Au", "Cu", "Density")
          - overall: {length_weighted_mean, max, min, count, accumulation_grade_meters, ...}
          - by_hole[]: array of {hole_id, length_weighted_mean, max_grade, min_grade,
                        sample_count, total_length, accumulation, ...}
          - hole_count: number of holes
      - gap_analysis: {total_gap_count, total_gap_length, holes_with_gaps, holes_without_gaps}

    Args:
        attribute: Filter to a specific attribute name (e.g. "Au"). Empty = all attributes.
        metric: Which overall metric to sort by. One of: length_weighted_mean, max, min,
                count, accumulation_grade_meters. Default: length_weighted_mean.
        min_value: Only return results where the metric >= this value. None = no filter.
        object_name: Filter to a specific object name (substring match). Empty = all objects.
        top_n: Number of top results to return (default 10).
        include_by_hole: If True, include the per-hole breakdown in results (can be large).

    Returns:
        A list of result dicts with object_name, collection_name, attribute,
        overall stats, and optionally by_hole data, sorted by the chosen metric descending.
    """
    # Build the aggregation pipeline
    match_stage: dict = {"data_type": "numeric", "doc_type": {"$in": ["complete", "summary"]}}

    if object_name:
        match_stage["object_name"] = {"$regex": object_name, "$options": "i"}

    pipeline = [
        {"$match": match_stage},
        {"$unwind": "$statistics"},
    ]

    # Filter by attribute if specified
    if attribute:
        pipeline.append({"$match": {"statistics.attribute": {"$regex": f"^{attribute}$", "$options": "i"}}})

    # Filter by minimum metric value
    metric_field = f"statistics.overall.{metric}"
    if min_value is not None:
        pipeline.append({"$match": {metric_field: {"$gte": min_value}}})

    # Sort by the chosen metric descending
    pipeline.append({"$sort": {metric_field: -1}})
    pipeline.append({"$limit": top_n})

    # Project the fields we want
    projection = {
        "object_name": 1,
        "object_type": 1,
        "collection_name": 1,
        "attribute": "$statistics.attribute",
        "overall": "$statistics.overall",
        "hole_count": "$statistics.hole_count",
    }
    if include_by_hole:
        projection["by_hole"] = "$statistics.by_hole"

    pipeline.append({"$project": projection})

    results = list(stats_collection.aggregate(pipeline))

    # Clean up for JSON serialization (remove ObjectId)
    cleaned = []
    for r in results:
        entry = {
            "object_name": r.get("object_name", "?"),
            "object_type": r.get("object_type", "?"),
            "collection_name": r.get("collection_name", "?"),
            "attribute": r.get("attribute", "?"),
            "overall": r.get("overall", {}),
            "hole_count": r.get("hole_count", 0),
        }
        if include_by_hole and "by_hole" in r:
            # Sort by_hole by LWM descending and limit to top 20 to keep response manageable
            by_hole = sorted(
                r.get("by_hole", []),
                key=lambda h: h.get("length_weighted_mean", 0),
                reverse=True,
            )[:20]
            entry["by_hole_top20"] = by_hole
        cleaned.append(entry)

    return cleaned


def list_objects_in_database() -> list[dict]:
    """List all distinct data objects stored in the statistics database.

    Returns a list of dicts with object_name, object_type, object_id,
    collection_names, and document count. Use this to discover what
    data is available before running targeted queries.
    """
    pipeline = [
        {"$group": {
            "_id": {"object_id": "$object_id", "object_name": "$object_name", "object_type": "$object_type"},
            "doc_count": {"$sum": 1},
            "collections": {"$addToSet": "$collection_name"},
            "attributes": {"$addToSet": "$statistics.attribute"},
        }},
        {"$sort": {"_id.object_name": 1}},
    ]
    results = []
    for row in stats_collection.aggregate(pipeline):
        info = row["_id"]
        # Flatten nested attribute lists
        flat_attrs = set()
        for attr_list in row.get("attributes", []):
            if isinstance(attr_list, list):
                flat_attrs.update(attr_list)
            elif isinstance(attr_list, str):
                flat_attrs.add(attr_list)
        results.append({
            "object_name": info.get("object_name", "?"),
            "object_type": info.get("object_type", "?"),
            "object_id": info.get("object_id", "?"),
            "collections": row.get("collections", []),
            "attributes": sorted(flat_attrs),
            "doc_count": row.get("doc_count", 0),
        })
    return results


query_tool = FunctionTool(func=query_statistics)
list_objects_tool = FunctionTool(func=list_objects_in_database)

# ---------------------------------------------------------------------------
# 4. RAG agent — hierarchy-aware semantic search + entity listing
#    (query_statistics is retained above but NOT exposed to the agent)
# ---------------------------------------------------------------------------

rag_instruction = """\
You are a geoscience data assistant with access to a MongoDB database of \
drillhole statistics and summaries organized in a four-tier hierarchy:

  ORGANISATION → WORKSPACE → OBJECT → COMPONENT

Each level describes a different entity (the organisation, a workspace, a data \
object, or a component) and has its own embedded summary optimized for different \
question scopes.

You have two tools:

1. **search_statistics** — semantic search over embedded summaries at ALL hierarchy \
   levels. Returns the most relevant summaries ranked by cosine similarity. Each result \
   includes a `hierarchy_level` field:
   - "organisation" — portfolio-wide overview across all workspaces
   - "workspace" — comparison across data objects within a single workspace \
     (data object inventory, grade comparisons, data scope)
   - "object" — cross-component comparison within one data object \
     (component inventory, key attributes, overall highlights)
   - "component" — detailed per-attribute, per-hole statistics for one \
     interval table (LWM, max, min, per-hole rankings, gap analysis)

2. **list_objects_in_database** — discover what data objects and attributes exist

<rules>
- Use search_statistics for ALL questions — the hierarchical embeddings will \
  naturally route your query to the most relevant entity level.
- For broad questions ("what data is available?", "which workspace has the most \
  data?"), the search naturally returns organisation or workspace entities.
- For data-object-scoped questions ("tell me about wolfpass", "compare components \
  in a specific data object"), the search returns object-level entities.
- For specific questions ("what is the Au LWM in the assays table?", "top holes \
  by grade in lithology"), the search returns component-level entities.
- Use a higher top_k (3-5) when the question spans multiple entities or is broad.
- Use a lower top_k (1-2) for specific single-component questions.
- If the first search doesn't give enough detail, refine your query to be more \
  specific (targeting component-level) or broader (targeting workspace/org level).
- If unsure what data exists, call list_objects_in_database first.
- Cite the data object name, component name, and hole IDs when referencing data.
- Include specific numbers (LWM, max grade, accumulation) in your answers.
- Pay attention to the hierarchy_level of each result — it tells you the entity scope.
- If no relevant results are found, say so honestly.
- Do NOT fabricate statistics that are not present in the tool results.
</rules>

<hierarchy_detail>
Organisation entities open with "ORGANISATION SUMMARY" and cover:
  - Total workspaces, total data objects, data portfolio scope
  - Cross-workspace comparison and strategic highlights

Workspace entities open with "WORKSPACE SUMMARY" and cover:
  - All data objects in the workspace, their types and components
  - Cross-entity comparisons (grades, hole counts, data volume)
  - Workspace-wide patterns and standout highlights

Object entities open with "OBJECT SUMMARY" and cover:
  - All interval tables (components) in one data object
  - Cross-component comparison (attributes, hole counts, grades)
  - Key attribute highlights and gap analysis across tables

Component entities cover:
  - Per-attribute statistics (LWM, max, min, count, accumulation)
  - Per-hole rankings (top 10 by LWM, top 5 by max grade)
  - Gap analysis and data quality for one interval table
</hierarchy_detail>
"""

rag_agent = LlmAgent(
    model=EVO_AGENT_MODEL,
    name="stats_rag_agent",
    description="Answers geoscience questions by querying a four-tier hierarchical database of drillhole statistics.",
    instruction=rag_instruction,
    tools=[search_tool, list_objects_tool],
)

RAG_APP_NAME = "stats_rag_agent"
rag_runner = InMemoryRunner(agent=rag_agent, app_name=RAG_APP_NAME)

print(f"RAG agent ready — model: {EVO_AGENT_MODEL}")
print(f"Tools: search_statistics, list_objects_in_database")
print(f"Documents in DB: {stats_collection.count_documents({})}")
embed_count = stats_collection.count_documents({"agent_summary_embedding": {"$exists": True}})
print(f"  with embeddings: {embed_count}")
for level in ["organisation", "workspace", "object", "component"]:
    n = stats_collection.count_documents({"hierarchy_level": level, "agent_summary_embedding": {"$exists": True}})
    print(f"    {level}: {n}")

RAG agent ready — model: gemini-3-flash-preview
Tools: search_statistics, list_objects_in_database
Documents in DB: 10
  with embeddings: 10
    organisation: 1
    workspace: 1
    object: 3
    component: 5


In [25]:
# --- Ask the RAG agent a question ---
QUESTION1 = "Which objects have the highest gold grades and what are the top holes?"
QUESTION2 = "Which objects holes are the best?"

QUESTION = QUESTION2

rag_session = await rag_runner.session_service.create_session(
    app_name=RAG_APP_NAME, user_id="notebook_user"
)

response_parts = []
tool_calls = []
rag_response = rag_runner.run_async(
    user_id="notebook_user",
    session_id=rag_session.id,
    new_message=types.Content(
        role="user",
        parts=[types.Part(text=QUESTION)],
    ),
)

print(f"Q: {QUESTION}\n")
async for event in rag_response:
    # Log the raw event author for debugging
    author = getattr(event, "author", "?")

    if event.content and event.content.parts:
        for part in event.content.parts:
            # --- Tool call (request) ---
            fc = getattr(part, "function_call", None)
            if fc:
                args_str = json.dumps(fc.args, default=str) if fc.args else "{}"
                tool_calls.append({"tool": fc.name, "args": fc.args})
                print(f"  [{author}] tool_call → {fc.name}({args_str})")

            # --- Tool response ---
            fr = getattr(part, "function_response", None)
            if fr:
                resp = fr.response
                if isinstance(resp, dict):
                    # Show result count or keys
                    preview = f"{len(resp.get('result', resp))} items" if "result" in resp else list(resp.keys())
                elif isinstance(resp, list):
                    preview = f"{len(resp)} results"
                else:
                    preview = str(resp)[:200]
                print(f"  [{author}] tool_result ← {fr.name}: {preview}")

            # --- Text output ---
            if part.text:
                response_parts.append(part.text)

rag_answer = "".join(response_parts)

# --- Telemetry summary ---
print(f"\n{'─'*60}")
print(f"Telemetry: {len(tool_calls)} tool call(s)")
for i, tc in enumerate(tool_calls, 1):
    print(f"  {i}. {tc['tool']}({json.dumps(tc['args'], default=str)})")
print(f"Answer length: {len(rag_answer)} chars")
print(f"{'─'*60}\n")

print(f"A:\n{rag_answer}")

Q: Which objects holes are the best?

  [stats_rag_agent] tool_call → search_statistics({"top_k": 3, "query": "Which data objects or workspaces have the best drillholes in terms of grade and mineralisation?"})
  [WORKSPACE] workspace / ? (workspace_summary): 0.7422
  [ORGANISATION] organisation / ? (organisation_summary): 0.7004
  [OBJECT] Maia Drillholes / ? (object_summary): 0.6845
  [COMPONENT] component / assay (complete): 0.6844
  [OBJECT] Wolfpass Drillhole Data / ? (object_summary): 0.6630
  [OBJECT] Maia Geology (Desurveyed) / ? (object_summary): 0.6571
  [COMPONENT] component / assay (complete): 0.6510
  [COMPONENT] component / geology (complete): 0.6449
  [COMPONENT] component / intervals (complete): 0.6368
  [COMPONENT] component / lithology (complete): 0.6295
  [stats_rag_agent] tool_result ← search_statistics: 1 items

────────────────────────────────────────────────────────────
Telemetry: 1 tool call(s)
  1. search_statistics({"top_k": 3, "query": "Which data objects or w

## 7. Verify and Query MongoDB

Query the collection to verify the data was stored correctly and explore historical statistics.

In [ ]:
# Collection overview
doc_count = stats_collection.count_documents({})
print(f"Total documents in collection: {doc_count}\n")

# Count by doc_type
for doc_type in ["complete", "summary", "detail"]:
    count = stats_collection.count_documents({"doc_type": doc_type})
    if count > 0:
        print(f"  {doc_type}: {count} document(s)")

# Show documents for all processed objects
for obj_id in inserted_object_ids:
    print(f"\n{'─'*60}")
    print(f"Documents for object {obj_id}:")
    cursor = stats_collection.find(
        {"object_id": obj_id},
        {
            "collection_name": 1, "doc_type": 1, "data_type": 1,
            "statistics": 1, "gap_analysis": 1, "timestamp": 1,
            "metadata.doc_size_bytes": 1, "object_name": 1,
            "agent_summary": 1, "agent_summary_embedding": 1,
        }
    ).sort([("collection_name", 1), ("doc_type", 1)])

    for doc in cursor:
        coll = doc.get("collection_name", "?")
        dtype = doc.get("doc_type", "?")
        data_type = doc.get("data_type", "?")
        obj_name = doc.get("object_name", "?")
        size_kb = doc.get("metadata", {}).get("doc_size_bytes", 0) / 1024
        stats = doc.get("statistics", [])
        gaps = doc.get("gap_analysis", {}).get("total_gap_count", 0)
        
        if dtype in ("complete", "summary"):
            # Embedding info
            embedding = doc.get("agent_summary_embedding")
            embed_info = f", embed {len(embedding)}d" if embedding else ", no embedding"
            summary = doc.get("agent_summary", "")
            summary_info = f", summary {len(summary)} chars" if summary else ""
            print(f"\n  [{obj_name}] {coll} [{dtype}] {data_type} - {len(stats)} attributes, {gaps} gaps, {size_kb:.0f} KB{summary_info}{embed_info}")
            
            if data_type == "numeric":
                # Show top 5 numeric attributes by LWM
                sorted_stats = sorted(stats, key=lambda s: abs(s.get("overall", {}).get("length_weighted_mean") or 0), reverse=True)
                for s in sorted_stats[:5]:
                    ov = s.get("overall", {})
                    print(f"     {s['attribute']}: LWM={ov.get('length_weighted_mean', 'N/A')}, max={ov.get('max', 'N/A')}, n={ov.get('count', 'N/A')}")
                if len(sorted_stats) > 5:
                    print(f"     ... and {len(sorted_stats) - 5} more")
            else:
                # Show categorical attribute summaries
                for cs in stats:
                    ov = cs.get("overall", {})
                    top_vals = [v["value"] for v in ov.get("value_counts", [])[:3]]
                    print(f"     {cs['attribute']}: {ov.get('unique_count', 0)} unique values, "
                          f"top: {', '.join(top_vals)}")
        else:
            attrs = doc.get("attributes_in_chunk", [])
            print(f"\n  [{obj_name}] {coll} [{dtype}] - {len(attrs)} attributes (detail chunk), {size_kb:.0f} KB")

Total documents in collection: 3

  complete: 3 document(s)

────────────────────────────────────────────────────────────
Documents for object 0286ea01-1a2c-41a8-81b8-fca7df38feac:

  [Maia Drillholes] assay [complete] numeric - 1 attributes, 0 gaps, 1 KB
     Au: LWM=0.4725068505663135, max=5.91, n=2210

  [Maia Drillholes] geology [complete] categorical - 1 attributes, 0 gaps, 3 KB
     Lithology: 5 unique values, top: MiS, Casing, QzP

────────────────────────────────────────────────────────────
Documents for object ea6ced6a-31d2-44a9-b8ae-1cee0b5ea42e:

  [Maia Geology (Desurveyed)] intervals [complete] categorical - 1 attributes, 0 gaps, 3 KB
     Lithology: 5 unique values, top: MiS, Casing, QzP


## 8. Query High-Grade Objects

Example queries for finding objects by grade statistics.

In [ ]:
# Example queries - uncomment to run:

# 1. Find all objects with Au length-weighted mean >= 1.0 g/t
high_au_objects = find_high_grade_objects(stats_collection, grade="Au", min_lwm=0.4)
print(f"Found {len(high_au_objects)} objects with Au LWM >= 1.0")

# 2. Find objects with Au peak values >= 10 g/t
# high_peak_objects = find_high_grade_objects(stats_collection, grade="Au", min_max=10.0)

# 3. Top 10 objects by Au length-weighted mean
# top_au = get_top_objects_by_grade(stats_collection, grade="Au", metric="lwm", top_n=10)
# for obj in top_au:
#     print(f"  {obj['object_name']}: {obj['lwm']:.4f}")

# 4. Find high-grade objects in a specific workspace
# workspace_high_grade = find_high_grade_objects(
#     stats_collection, 
#     grade="Au", 
#     min_lwm=0.5, 
#     workspace_id=WORKSPACE_ID
# )

Found 1 objects with Au LWM >= 1.0


## 9. Cascade Update

Bidirectional cascade: re-process components **downward** then propagate summaries **upward**.

```
       ┌─── downward: re-download, recalculate stats, re-summarize components
       ▼
  COMPONENT ──► OBJECT ──► WORKSPACE ──► ORGANISATION
                 ▲
                 └─── upward: re-generate summary + embedding at each level from children
```

**Modes:**
| Config | Behaviour |
|---|---|
| `CHANGED_OBJECT_ID = "uuid"` | Full reprocess: download → stats → upsert components → cascade up |
| `CHANGED_COMPONENT_ID = ObjectId(...)` | Upward cascade only (no re-download) |
| Both `None` | Full workspace refresh — cascade up for all objects |

**How it works:**
1. **Downward** (`reprocess_object`): Loads the Evo object, downloads interval tables, recalculates statistics, upserts component docs (stable `_id`s via unique index), and re-runs the component summarization agent
2. **Upward** (`cascade_up`): Walks `parent_id` links from the starting level to the root, re-running the appropriate summarization agent at each level using `get_children()` to gather sibling summaries
3. `parent_id` links are stable because all hierarchy docs use `update_one(upsert=True)`

In [ ]:
import re
import time
from datetime import datetime, timezone

# ── Configuration ─────────────────────────────────────────────────────────
# Mode 1: Set an Evo object UUID → full reprocess (down + up cascade)
CHANGED_OBJECT_ID = None   # e.g. "01c54ab3-..."

# Mode 2: Set a MongoDB _id → upward cascade only (no re-download)
CHANGED_COMPONENT_ID = None  # e.g. ObjectId("6654...")

# If both None → full workspace refresh (cascade up for all objects)


# ══════════════════════════════════════════════════════════════════════════
# Downward cascade: reprocess a single object's components
# ══════════════════════════════════════════════════════════════════════════

async def reprocess_object(obj_id: str, workspace_id: str) -> list[dict]:
    """Full re-process pipeline for one Evo object.

    Steps: load → download → stats → prepare → upsert → summarize.
    Returns the list of upserted component documents (with stable _ids).
    """
    t0 = time.perf_counter()

    # --- 1. Load object from Evo ---
    obj, obj_dict, object_name, object_type, collections_info = await load_downhole_object(
        workspace_id, obj_id, VERSION
    )
    print(f"  Loaded: {object_name} ({object_type}), {len(collections_info)} table(s)")

    # --- 2. Download interval tables ---
    collection_data = await download_all_interval_tables(obj, object_type, collections_info)
    for coll_name, coll_data in collection_data.items():
        df = coll_data["df"]
        print(f"    {coll_name}: {len(df):,} intervals, "
              f"{len(coll_data['numeric_cols'])} numeric / {len(coll_data['categorical_cols'])} categorical")

    # --- 3. Calculate statistics ---
    all_collection_stats = {}
    for coll_name, coll_data in collection_data.items():
        df = coll_data["df"]
        numeric_cols = coll_data["numeric_cols"]
        categorical_cols = coll_data["categorical_cols"]
        attribute_stats = {}

        for grade_col in numeric_cols:
            try:
                overall = calculate_interval_statistics(df, grade_col)
            except (ValueError, ZeroDivisionError):
                continue
            hole_stats_df = calculate_statistics_by_hole(df, grade_col)
            attribute_stats[grade_col] = {
                "overall": overall,
                "hole_count": len(hole_stats_df),
                "by_hole": hole_stats_df.to_dict(orient="records"),
            }

        categorical_stats = {}
        for cat_col in categorical_cols:
            try:
                cat_stats = calculate_categorical_statistics(df, cat_col)
                categorical_stats[cat_col] = cat_stats
            except Exception:
                pass

        gap_analysis = analyze_gaps(df)
        all_collection_stats[coll_name] = {
            "attributes": attribute_stats,
            "categorical": categorical_stats,
            "gap_analysis": gap_analysis,
        }

    total_attrs = sum(len(v["attributes"]) for v in all_collection_stats.values())
    print(f"  {total_attrs} numeric attributes across {len(all_collection_stats)} table(s)")

    # --- 4. Prepare MongoDB documents ---
    all_documents = []
    for coll_name, coll_stats in all_collection_stats.items():
        docs = prepare_collection_documents(
            workspace_id=workspace_id,
            object_id=obj_id,
            object_name=object_name,
            object_type=object_type,
            collection_name=coll_name,
            attribute_stats=coll_stats["attributes"],
            gap_analysis=coll_stats["gap_analysis"],
            categorical_stats=coll_stats["categorical"],
        )
        all_documents.extend(docs)

    # --- 5. Upsert component docs (stable _ids via unique index) ---
    for doc in all_documents:
        doc["hierarchy_level"] = "component"
        comp_filter = {
            "hierarchy_level": "component",
            "object_id": obj_id,
            "collection_name": doc["collection_name"],
            "doc_type": doc["doc_type"],
            "data_type": doc.get("data_type", ""),
        }
        result = stats_collection.update_one(comp_filter, {"$set": doc}, upsert=True)
        doc["_id"] = result.upserted_id or stats_collection.find_one(comp_filter, {"_id": 1})["_id"]

    print(f"  Upserted {len(all_documents)} component doc(s)")

    # Build the parent object URI (all components share the same parent object)
    comp_uri = build_object_uri(
        EVO_WEB_BASE_URL, str(evo_context.org_id), workspace_id, obj_id,
    )

    # --- 6. Re-summarize each component with the ADK agent ---
    for doc_idx, doc in enumerate(all_documents, 1):
        doc_for_prompt = {k: v for k, v in doc.items() if k != "_id"}
        doc_json = json.dumps(doc_for_prompt, indent=2, default=str)

        doc_session = await runner.session_service.create_session(
            app_name=APP_NAME, user_id="notebook_user"
        )
        summary_parts = []
        agent_response = runner.run_async(
            user_id="notebook_user",
            session_id=doc_session.id,
            new_message=types.Content(
                role="user",
                parts=[types.Part(text=(
                    "Produce a detailed summary of the following statistics. "
                    "Include ALL overall values and analyse the per-hole (by_hole) data — "
                    "rank the top 10 holes by length-weighted mean, top 5 by max grade, "
                    "and note holes with highest accumulation and longest sampled length.\n\n"
                    f"{doc_json}"
                ))]
            ),
        )
        async for event in agent_response:
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        summary_parts.append(part.text)

        summary_text = "".join(summary_parts)
        summary_embedding = get_embedding(summary_text)

        stats_collection.update_one(
            {"_id": doc["_id"]},
            {"$set": {
                "agent_summary": summary_text,
                "agent_summary_embedding": summary_embedding,
                "referenced_uris": [comp_uri],
            }}
        )

        coll_name = doc.get("collection_name", "?")
        doc_type = doc.get("doc_type", "?")
        print(f"    [{doc_idx}/{len(all_documents)}] {coll_name} [{doc_type}] — "
              f"{len(summary_text)} chars, {len(summary_embedding)}d embedding")

    elapsed = time.perf_counter() - t0
    print(f"  Reprocessed {len(all_documents)} component(s) in {elapsed:.1f}s")
    return all_documents


# ══════════════════════════════════════════════════════════════════════════
# Upward cascade: re-summarize a hierarchy doc from its children
# ══════════════════════════════════════════════════════════════════════════

async def _resummarize_level(
    doc: dict,
    stats_collection,
    runner_cache: dict,
) -> dict:
    """Re-generate the agent summary + embedding for *doc* using its children.

    Returns the updated doc dict (already upserted into MongoDB).
    """
    level = doc["hierarchy_level"]
    doc_id = doc["_id"]

    children = get_children(stats_collection, doc_id)
    if not children:
        print(f"    [{level.upper()}] _id={doc_id}: no children — skipping")
        return doc

    child_context = []
    for child in children:
        child_context.append({
            "hierarchy_level": child.get("hierarchy_level"),
            "object_name": child.get("object_name", ""),
            "collection_name": child.get("collection_name", ""),
            "doc_type": child.get("doc_type", ""),
            "agent_summary": child.get("agent_summary", ""),
        })

    if level == "object":
        instruction = object_instruction
        agent_name = "object_summary_agent"
        prompt_prefix = (
            "Produce an object-level summary for the following data object. "
            "Synthesize the individual component summaries into a cohesive entity overview.\n\n"
        )
        context_json = json.dumps({
            "object_name": doc.get("object_name", "?"),
            "object_type": doc.get("object_type", "?"),
            "object_id": doc.get("object_id", "?"),
            "collection_count": len(children),
            "collections": child_context,
        }, indent=2, default=str)
    elif level == "workspace":
        instruction = workspace_instruction
        agent_name = "workspace_summary_agent"
        prompt_prefix = (
            "Produce a workspace-level summary for the following workspace. "
            "Synthesize the individual object summaries into a cohesive entity overview.\n\n"
        )
        context_json = json.dumps({
            "workspace_id": doc.get("workspace_id", "?"),
            "object_count": len(children),
            "objects": child_context,
        }, indent=2, default=str)
    elif level == "organisation":
        instruction = org_instruction
        agent_name = "org_summary_agent"
        prompt_prefix = (
            "Produce an organisation-level summary of the entire drilling data portfolio. "
            "Synthesize the workspace summaries into a high-level entity overview.\n\n"
        )
        context_json = json.dumps({
            "workspace_count": len(children),
            "total_objects": sum(c.get("object_count", 0) for c in children),
            "workspaces": child_context,
        }, indent=2, default=str)
    else:
        print(f"    [{level.upper()}] unsupported level — skipping")
        return doc

    if agent_name not in runner_cache:
        agent = LlmAgent(
            model=EVO_AGENT_MODEL,
            name=agent_name,
            description=f"Produces {level}-level summaries.",
            instruction=instruction,
        )
        runner_cache[agent_name] = InMemoryRunner(agent=agent, app_name=agent_name)

    level_runner = runner_cache[agent_name]
    session = await level_runner.session_service.create_session(
        app_name=agent_name, user_id="notebook_user"
    )

    parts = []
    response = level_runner.run_async(
        user_id="notebook_user",
        session_id=session.id,
        new_message=types.Content(
            role="user",
            parts=[types.Part(text=prompt_prefix + context_json)],
        ),
    )
    async for event in response:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    parts.append(part.text)

    summary_text = "".join(parts)
    summary_embedding = get_embedding(summary_text)

    # ── Compute referenced_uris based on hierarchy level ──
    referenced_uris: list[str] = []
    if level == "object":
        # Object doc references itself
        obj_id = doc.get("object_id", "")
        ws_id = doc.get("workspace_id", WORKSPACE_ID)
        if obj_id:
            referenced_uris = [build_object_uri(
                EVO_WEB_BASE_URL, str(evo_context.org_id), ws_id, obj_id,
            )]
    elif level == "workspace":
        # Scan summary for object names mentioned in children
        ws_id = doc.get("workspace_id", WORKSPACE_ID)
        objects_map = {
            c.get("object_name", ""): c.get("object_id", "")
            for c in children
            if c.get("object_name") and c.get("object_id")
        }
        referenced_uris = extract_referenced_uris(
            summary_text, objects_map,
            EVO_WEB_BASE_URL, str(evo_context.org_id), ws_id,
        )
    elif level == "organisation":
        # Scan summary for object names across all workspaces
        all_obj_docs = list(stats_collection.find(
            {"hierarchy_level": "object", "object_name": {"$exists": True}},
            {"object_name": 1, "object_id": 1, "workspace_id": 1},
        ))
        for odoc in all_obj_docs:
            oname = odoc.get("object_name", "")
            oid = odoc.get("object_id", "")
            ows = odoc.get("workspace_id", "")
            if oname and oid and ows and re.search(re.escape(oname), summary_text, re.IGNORECASE):
                referenced_uris.append(
                    build_object_uri(EVO_WEB_BASE_URL, str(evo_context.org_id), ows, oid)
                )
        referenced_uris = list(dict.fromkeys(referenced_uris))

    update_fields = {
        "agent_summary": summary_text,
        "agent_summary_embedding": summary_embedding,
        "referenced_uris": referenced_uris,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    stats_collection.update_one({"_id": doc_id}, {"$set": update_fields})

    print(f"    [{level.upper()}] _id={doc_id}: {len(summary_text)} chars, "
          f"{len(summary_embedding)}d embedding ({len(children)} children, "
          f"{len(referenced_uris)} URIs)")

    doc.update(update_fields)
    return doc


async def cascade_up(start_id, stats_collection, include_start=False):
    """Walk parent_id chain from *start_id* upward, re-summarizing each level.

    Args:
        start_id: MongoDB _id to begin from.
        include_start: If True, re-summarize the starting doc too
                       (useful when starting from object level after
                       its components were refreshed).
    """
    chain = get_ancestor_chain(stats_collection, start_id)
    if not chain:
        print(f"  _id={start_id}: not found")
        return
    if len(chain) < 2 and not include_start:
        print(f"  _id={start_id}: no parent_id link — "
              "run the hierarchy cell (Cell 18) first to establish links.")
        return

    docs_to_update = chain if include_start else chain[1:]
    # Skip component level — components are re-summarized by reprocess_object
    docs_to_update = [d for d in docs_to_update if d["hierarchy_level"] != "component"]

    runner_cache = {}
    print(f"  Chain: {' → '.join(d['hierarchy_level'].upper() for d in chain)}")
    print(f"  Updating: {', '.join(d['hierarchy_level'].upper() for d in docs_to_update)}")
    for doc in docs_to_update:
        await _resummarize_level(doc, stats_collection, runner_cache)


# ══════════════════════════════════════════════════════════════════════════
# Execute
# ══════════════════════════════════════════════════════════════════════════

t_cascade_start = time.perf_counter()

if CHANGED_OBJECT_ID is not None:
    # ── Mode 1: Full reprocess (down + up) ────────────────────────────
    print(f"Mode 1: Reprocessing object {CHANGED_OBJECT_ID}\n")

    # Downward: re-download, recalculate, upsert components, re-summarize
    updated_components = await reprocess_object(CHANGED_OBJECT_ID, WORKSPACE_ID)

    # Upward: find the object-level doc and cascade up (including the object itself)
    obj_doc = stats_collection.find_one(
        {"hierarchy_level": "object", "object_id": CHANGED_OBJECT_ID},
        {"_id": 1},
    )
    if obj_doc:
        print(f"\n  Cascading upward from OBJECT...")
        await cascade_up(obj_doc["_id"], stats_collection, include_start=True)
    else:
        print(f"\n  No object-level doc found — run Cell 18 first to create hierarchy docs.")

elif CHANGED_COMPONENT_ID is not None:
    # ── Mode 2: Upward cascade only ──────────────────────────────────
    print(f"Mode 2: Cascading upward from component _id={CHANGED_COMPONENT_ID}\n")
    await cascade_up(CHANGED_COMPONENT_ID, stats_collection, include_start=False)

else:
    # ── Mode 3: Full workspace refresh ────────────────────────────────
    # Find all distinct objects that have hierarchy docs and cascade each
    obj_docs = list(stats_collection.find(
        {"hierarchy_level": "object", "workspace_id": WORKSPACE_ID},
        {"_id": 1, "object_name": 1, "object_id": 1},
    ))

    if not obj_docs:
        print("No object-level hierarchy docs found.\n"
              "Run Cell 18 first to create hierarchy docs and backfill parent_id.")
    else:
        print(f"Mode 3: Full workspace refresh — {len(obj_docs)} object(s)\n")
        for odoc in obj_docs:
            obj_name = odoc.get("object_name", odoc["object_id"])
            print(f"\n{'─'*60}")
            print(f"Object: {obj_name}")
            await cascade_up(odoc["_id"], stats_collection, include_start=True)

t_cascade_elapsed = time.perf_counter() - t_cascade_start

print(f"\n{'='*60}")
print(f"Cascade complete in {t_cascade_elapsed:.1f}s")
for level in ["component", "object", "workspace", "organisation"]:
    count = stats_collection.count_documents({"hierarchy_level": level})
    with_parent = stats_collection.count_documents({
        "hierarchy_level": level, "parent_id": {"$exists": True},
    })
    with_uris = stats_collection.count_documents({
        "hierarchy_level": level, "referenced_uris": {"$exists": True, "$ne": []},
    })
    print(f"  {level}: {count} docs ({with_parent} with parent_id, {with_uris} with URIs)")